<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/11_ECG_Noise_Robustness_And_Signal_Quality_Generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# NOTEBOOK 11 — NOISE ROBUSTNESS & SIGNAL-QUALITY GENERALIZATION
# STEP 11A — ENVIRONMENT & FROZEN BASELINE VERIFICATION
# ============================================================

!pip install wfdb -q
from google.colab import drive
drive.mount("/content/drive")
import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import wfdb

print("=" * 70)
print("STEP 11A — ENVIRONMENT & FROZEN BASELINE VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_PATH_11A = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

DATA_PATH_11A = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH_11A = (
    "/content/drive/MyDrive/PTB-XL Research Project/results"
)

DEEP_LEARNING_PATH_11A = os.path.join(
    RESULTS_PATH_11A,
    "deep_learning"
)

NOISE_RESULTS_PATH_11A = os.path.join(
    RESULTS_PATH_11A,
    "noise_robustness"
)

FROZEN_BASELINE_PATH_11A = os.path.join(
    DEEP_LEARNING_PATH_11A,
    "ecg_resnet1d_frozen_baseline.json"
)

CHECKPOINT_PATH_11A = os.path.join(
    DEEP_LEARNING_PATH_11A,
    "ecg_resnet1d_best_validation.pt"
)

TEST_RESULTS_PATH_11A = os.path.join(
    DEEP_LEARNING_PATH_11A,
    "ecg_resnet1d_test_results.csv"
)

# ------------------------------------------------------------
# 2. Verify paths
# ------------------------------------------------------------

required_paths_11A = {
    "project": PROJECT_PATH_11A,
    "dataset": DATA_PATH_11A,
    "results": RESULTS_PATH_11A,
    "deep_learning_results": DEEP_LEARNING_PATH_11A,
    "frozen_baseline": FROZEN_BASELINE_PATH_11A,
    "checkpoint": CHECKPOINT_PATH_11A,
    "test_results": TEST_RESULTS_PATH_11A
}

print("\nPATH VERIFICATION")
print("-" * 70)

for name_11A, path_11A in required_paths_11A.items():

    exists_11A = os.path.exists(path_11A)

    print(
        f"{name_11A:<25}: "
        f"{'PASS' if exists_11A else 'FAIL'}"
    )

    if not exists_11A:
        raise FileNotFoundError(
            f"Required path not found: {path_11A}"
        )

# ------------------------------------------------------------
# 3. Create Notebook 11 results directory
# ------------------------------------------------------------

os.makedirs(
    NOISE_RESULTS_PATH_11A,
    exist_ok=True
)

print(
    "\nNoise robustness results directory: "
    f"{NOISE_RESULTS_PATH_11A}"
)

# ------------------------------------------------------------
# 4. Load frozen baseline record
# ------------------------------------------------------------

with open(
    FROZEN_BASELINE_PATH_11A,
    "r"
) as file_11A:

    frozen_baseline_11A = json.load(
        file_11A
    )

print("\nFROZEN BASELINE")
print("-" * 70)

print(
    f"Status:                   "
    f"{frozen_baseline_11A['status']}"
)

print(
    f"Model:                    "
    f"{frozen_baseline_11A['model']}"
)

print(
    f"Architecture:             "
    f"{frozen_baseline_11A['architecture']}"
)

print(
    f"Representation:           "
    f"{frozen_baseline_11A['input_representation']}"
)

print(
    f"Parameters:               "
    f"{frozen_baseline_11A['parameter_count']:,}"
)

print(
    f"Selected checkpoint:      "
    f"Epoch "
    f"{frozen_baseline_11A['selected_checkpoint_epoch']}"
)

print(
    f"Validation macro-AUROC:   "
    f"{frozen_baseline_11A['validation_macro_AUROC']:.6f}"
)

print(
    f"Locked test macro-AUROC:  "
    f"{frozen_baseline_11A['test_macro_AUROC']:.6f}"
)

# ------------------------------------------------------------
# 5. Verify frozen baseline properties
# ------------------------------------------------------------

assert frozen_baseline_11A["status"] == "FROZEN"

assert frozen_baseline_11A[
    "model"
] == "ECGResNet1D"

assert frozen_baseline_11A[
    "architecture"
] == "1D ResNet"

assert frozen_baseline_11A[
    "input_representation"
] == "raw 12-lead ECG waveform"

assert frozen_baseline_11A[
    "input_shape"
] == [12, 5000]

assert frozen_baseline_11A[
    "sampling_frequency_hz"
] == 500

assert frozen_baseline_11A[
    "duration_seconds"
] == 10

assert frozen_baseline_11A[
    "number_of_targets"
] == 5

assert frozen_baseline_11A[
    "targets"
] == [
        "MI",
        "STTC",
        "CD",
        "HYP",
        "NORM"
    ]

assert frozen_baseline_11A[
    "parameter_count"
] == 8739973

assert frozen_baseline_11A[
    "selected_checkpoint_epoch"
] == 17

assert frozen_baseline_11A[
    "test_set_used_for_model_selection"
] is False

assert frozen_baseline_11A[
    "test_set_used_for_threshold_tuning"
] is False

assert frozen_baseline_11A[
    "test_set_used_for_hyperparameter_tuning"
] is False

# ------------------------------------------------------------
# 6. Verify checkpoint exists and is readable
# ------------------------------------------------------------

checkpoint_11A = torch.load(
    CHECKPOINT_PATH_11A,
    map_location="cpu"
)

assert isinstance(
    checkpoint_11A,
    dict
)

assert checkpoint_11A[
    "epoch"
] == 17

assert checkpoint_11A[
    "target_columns"
] == [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

assert checkpoint_11A[
    "model_parameters"
] == 8739973

print("\nCHECKPOINT VERIFICATION")
print("-" * 70)

print(
    f"Checkpoint epoch:        "
    f"{checkpoint_11A['epoch']}"
)

print(
    f"Checkpoint parameters:   "
    f"{checkpoint_11A['model_parameters']:,}"
)

print(
    f"Checkpoint val AUROC:    "
    f"{checkpoint_11A['best_validation_macro_auroc']:.6f}"
)

# ------------------------------------------------------------
# 7. Verify locked test results
# ------------------------------------------------------------

test_results_11A = pd.read_csv(
    TEST_RESULTS_PATH_11A
)

assert len(test_results_11A) == 5

assert set(
    test_results_11A["target"]
) == {
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
}

test_macro_auroc_11A = float(
    test_results_11A["AUROC"].mean()
)

assert abs(
    test_macro_auroc_11A - 0.930060
) < 1e-5

print("\nLOCKED TEST RESULT VERIFICATION")
print("-" * 70)

print(
    f"Test ECGs:               "
    f"2,198"
)

print(
    f"Test macro-AUROC:        "
    f"{test_macro_auroc_11A:.6f}"
)

# ------------------------------------------------------------
# 8. Environment
# ------------------------------------------------------------

print("\nENVIRONMENT")
print("-" * 70)

print(
    f"Python:                   "
    f"{__import__('sys').version.split()[0]}"
)

print(
    f"NumPy:                    "
    f"{np.__version__}"
)

print(
    f"Pandas:                   "
    f"{pd.__version__}"
)

print(
    f"PyTorch:                  "
    f"{torch.__version__}"
)

print(
    f"WFDB:                     "
    f"{wfdb.__version__}"
)

print(
    f"CUDA available:           "
    f"{torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"GPU:                      "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA version:             "
        f"{torch.version.cuda}"
    )

# ------------------------------------------------------------
# 9. Record checkpoint hash
# ------------------------------------------------------------

sha256_11A = hashlib.sha256()

with open(
    CHECKPOINT_PATH_11A,
    "rb"
) as file_11A:

    for chunk_11A in iter(
        lambda: file_11A.read(1024 * 1024),
        b""
    ):
        sha256_11A.update(
            chunk_11A
        )

checkpoint_hash_11A = sha256_11A.hexdigest()

print("\nFROZEN CHECKPOINT SHA-256")
print("-" * 70)
print(checkpoint_hash_11A)

# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 11A STATUS: PASS")
print("Frozen Notebook 10 baseline verified.")
print("No model training or test-set modification performed.")
print("=" * 70)

Mounted at /content/drive
STEP 11A — ENVIRONMENT & FROZEN BASELINE VERIFICATION

PATH VERIFICATION
----------------------------------------------------------------------
project                  : PASS
dataset                  : PASS
results                  : PASS
deep_learning_results    : PASS
frozen_baseline          : PASS
checkpoint               : PASS
test_results             : PASS

Noise robustness results directory: /content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness

FROZEN BASELINE
----------------------------------------------------------------------
Status:                   FROZEN
Model:                    ECGResNet1D
Architecture:             1D ResNet
Representation:           raw 12-lead ECG waveform
Parameters:               8,739,973
Selected checkpoint:      Epoch 17
Validation macro-AUROC:   0.934005
Locked test macro-AUROC:  0.930060

CHECKPOINT VERIFICATION
----------------------------------------------------------------------
Checkpoint epo

In [5]:
import os
import json
import numpy as np
import pandas as pd

# ============================================================
# STEP 11B — NOISE ROBUSTNESS EVALUATION PROTOCOL
# ============================================================

NOISE_RESULTS_PATH_11B = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

os.makedirs(NOISE_RESULTS_PATH_11B, exist_ok=True)

TARGET_COLUMNS_11B = ["MI", "STTC", "CD", "HYP", "NORM"]

TEST_N_11B = 2198
SAMPLING_FREQUENCY_11B = 500
SIGNAL_DURATION_SECONDS_11B = 10
SIGNAL_LENGTH_11B = 5000
N_LEADS_11B = 12

NOISE_LEVELS_11B = [
    {
        "condition": "clean",
        "noise_type": "none",
        "snr_db": None,
        "description": "Original unmodified ECG waveform"
    },
    {
        "condition": "snr_30db",
        "noise_type": "gaussian",
        "snr_db": 30,
        "description": "Additive Gaussian noise at 30 dB SNR"
    },
    {
        "condition": "snr_20db",
        "noise_type": "gaussian",
        "snr_db": 20,
        "description": "Additive Gaussian noise at 20 dB SNR"
    },
    {
        "condition": "snr_10db",
        "noise_type": "gaussian",
        "snr_db": 10,
        "description": "Additive Gaussian noise at 10 dB SNR"
    },
    {
        "condition": "snr_0db",
        "noise_type": "gaussian",
        "snr_db": 0,
        "description": "Additive Gaussian noise at 0 dB SNR"
    }
]

PROTOCOL_11B = {
    "purpose": (
        "Evaluate robustness of the frozen raw-waveform ResNet-1D "
        "under controlled additive Gaussian noise."
    ),
    "model": "ECGResNet1D",
    "model_status": "frozen",
    "checkpoint_epoch": 17,
    "representation": "raw 12-lead ECG waveform",
    "input_shape": [N_LEADS_11B, SIGNAL_LENGTH_11B],
    "sampling_frequency_hz": SAMPLING_FREQUENCY_11B,
    "duration_seconds": SIGNAL_DURATION_SECONDS_11B,
    "targets": TARGET_COLUMNS_11B,
    "test_ecg_count": TEST_N_11B,
    "split": "patient-independent frozen Notebook 10 test split",
    "noise_domain": "synthetic controlled perturbation",
    "noise_type": "additive zero-mean Gaussian noise",
    "snr_definition": (
        "10*log10(signal_power/noise_power), "
        "with noise power matched to the clean signal power"
    ),
    "noise_levels_db": [30, 20, 10, 0],
    "clean_condition_included": True,
    "model_retraining": False,
    "threshold_tuning": False,
    "test_set_used_for_model_selection": False,
    "random_seed": 42,
    "primary_metric": "macro_AUROC",
    "secondary_metrics": [
        "macro_AUPRC",
        "per_class_AUROC",
        "per_class_AUPRC",
        "per_class_F1",
        "per_class_sensitivity",
        "per_class_specificity",
        "degradation_from_clean"
    ],
    "interpretation": (
        "This experiment measures predictive robustness, not clinical "
        "performance or clinical noise tolerance."
    )
}

PROTOCOL_PATH_11B = os.path.join(
    NOISE_RESULTS_PATH_11B,
    "step11B_noise_robustness_protocol.json"
)

with open(PROTOCOL_PATH_11B, "w", encoding="utf-8") as f:
    json.dump(PROTOCOL_11B, f, indent=2)

print("=" * 70)
print("STEP 11B — NOISE ROBUSTNESS EVALUATION PROTOCOL")
print("=" * 70)
print()
print("Model:", PROTOCOL_11B["model"])
print("Model status:", PROTOCOL_11B["model_status"])
print("Checkpoint epoch:", PROTOCOL_11B["checkpoint_epoch"])
print("Test ECGs:", PROTOCOL_11B["test_ecg_count"])
print("Input shape:", PROTOCOL_11B["input_shape"])
print("Sampling frequency:", f'{PROTOCOL_11B["sampling_frequency_hz"]} Hz')
print()
print("Noise conditions:")

for condition in NOISE_LEVELS_11B:
    print(
        f'  {condition["condition"]:>10} | '
        f'{condition["description"]}'
    )

print()
print("Retraining:", PROTOCOL_11B["model_retraining"])
print("Threshold tuning:", PROTOCOL_11B["threshold_tuning"])
print(
    "Test used for model selection:",
    PROTOCOL_11B["test_set_used_for_model_selection"]
)
print()
print("Protocol saved to:")
print(PROTOCOL_PATH_11B)
print()
print("STEP 11B STATUS: PASS")

STEP 11B — NOISE ROBUSTNESS EVALUATION PROTOCOL

Model: ECGResNet1D
Model status: frozen
Checkpoint epoch: 17
Test ECGs: 2198
Input shape: [12, 5000]
Sampling frequency: 500 Hz

Noise conditions:
       clean | Original unmodified ECG waveform
    snr_30db | Additive Gaussian noise at 30 dB SNR
    snr_20db | Additive Gaussian noise at 20 dB SNR
    snr_10db | Additive Gaussian noise at 10 dB SNR
     snr_0db | Additive Gaussian noise at 0 dB SNR

Retraining: False
Threshold tuning: False
Test used for model selection: False

Protocol saved to:
/content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness/step11B_noise_robustness_protocol.json

STEP 11B STATUS: PASS


In [6]:
import os
import numpy as np
import pandas as pd
import wfdb
from scipy import signal

# ============================================================
# STEP 11C — PTB-XL NATURAL SIGNAL-QUALITY CHARACTERIZATION
# ============================================================

DATA_PATH_11C = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH_11C = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

SPLIT_PATH_11C = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/ptbxl_patient_independent_split.csv"
)

METADATA_PATH_11C = os.path.join(
    DATA_PATH_11C,
    "ptbxl_database.csv"
)

QUALITY_OUTPUT_PATH_11C = os.path.join(
    RESULTS_PATH_11C,
    "step11C_test_signal_quality.csv"
)

# ------------------------------------------------------------
# Load frozen patient-independent split
# ------------------------------------------------------------

split_df_11C = pd.read_csv(SPLIT_PATH_11C)

test_ids_11C = (
    split_df_11C.loc[
        split_df_11C["split"] == "test",
        "ecg_id"
    ]
    .astype(int)
    .tolist()
)

assert len(test_ids_11C) == 2198
assert len(set(test_ids_11C)) == 2198

# ------------------------------------------------------------
# Load PTB-XL metadata
# ------------------------------------------------------------

meta_11C = pd.read_csv(METADATA_PATH_11C)

meta_11C["ecg_id"] = meta_11C["ecg_id"].astype(int)

test_meta_11C = (
    meta_11C[
        meta_11C["ecg_id"].isin(test_ids_11C)
    ]
    .copy()
)

assert len(test_meta_11C) == 2198

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def rms_11C(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.sqrt(np.mean(x ** 2)))


def baseline_wander_ratio_11C(x, fs=500):
    """
    Estimate low-frequency baseline content using a
    0.5 Hz low-pass Butterworth filter.

    Ratio = RMS(low-frequency component) / RMS(signal).
    This is a descriptive signal-quality measure, not
    a clinical quality label.
    """
    x = np.asarray(x, dtype=np.float64)

    total_rms = rms_11C(x)

    if total_rms == 0:
        return 0.0

    b, a = signal.butter(
        4,
        0.5,
        btype="lowpass",
        fs=fs
    )

    low_freq = signal.filtfilt(
        b,
        a,
        x
    )

    return float(rms_11C(low_freq) / total_rms)


def high_frequency_ratio_11C(x, fs=500, cutoff=40):
    """
    Estimate high-frequency content above 40 Hz.

    Ratio = RMS(high-frequency component) / RMS(signal).
    """
    x = np.asarray(x, dtype=np.float64)

    total_rms = rms_11C(x)

    if total_rms == 0:
        return 0.0

    b, a = signal.butter(
        4,
        cutoff,
        btype="highpass",
        fs=fs
    )

    high_freq = signal.filtfilt(
        b,
        a,
        x
    )

    return float(rms_11C(high_freq) / total_rms)


def signal_range_11C(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.ptp(x))


# ------------------------------------------------------------
# Characterize every test ECG
# ------------------------------------------------------------

rows_11C = []

print("=" * 70)
print("STEP 11C — NATURAL PTB-XL SIGNAL-QUALITY CHARACTERIZATION")
print("=" * 70)
print()
print(f"Test ECGs: {len(test_ids_11C)}")
print("Leads per ECG: 12")
print("Sampling frequency: 500 Hz")
print()

for i, (_, row) in enumerate(test_meta_11C.iterrows(), start=1):

    ecg_id = int(row["ecg_id"])
    filename = row["filename_hr"]

    record_path = os.path.join(
        DATA_PATH_11C,
        filename
    )

    record = wfdb.rdrecord(record_path)

    x = np.asarray(
        record.p_signal,
        dtype=np.float64
    )

    assert x.shape == (5000, 12), (
        f"Unexpected shape for ECG {ecg_id}: {x.shape}"
    )

    assert np.isfinite(x).all(), (
        f"Non-finite values found in ECG {ecg_id}"
    )

    lead_rms = []
    lead_ranges = []
    lead_bw = []
    lead_hf = []

    for lead_idx in range(12):

        lead = x[:, lead_idx]

        lead_rms.append(rms_11C(lead))
        lead_ranges.append(signal_range_11C(lead))
        lead_bw.append(
            baseline_wander_ratio_11C(lead)
        )
        lead_hf.append(
            high_frequency_ratio_11C(lead)
        )

    rows_11C.append({
        "ecg_id": ecg_id,
        "patient_id": row["patient_id"],
        "mean_lead_rms_mV": float(np.mean(lead_rms)),
        "median_lead_rms_mV": float(np.median(lead_rms)),
        "mean_lead_range_mV": float(np.mean(lead_ranges)),
        "median_lead_range_mV": float(np.median(lead_ranges)),
        "mean_baseline_wander_ratio": float(np.mean(lead_bw)),
        "max_baseline_wander_ratio": float(np.max(lead_bw)),
        "mean_high_frequency_ratio": float(np.mean(lead_hf)),
        "max_high_frequency_ratio": float(np.max(lead_hf)),
        "zero_rms_leads": int(
            np.sum(np.asarray(lead_rms) == 0)
        )
    })

    if i % 100 == 0 or i == len(test_meta_11C):
        print(
            f"Processed {i:4d}/{len(test_meta_11C)} ECGs"
        )

# ------------------------------------------------------------
# Save result
# ------------------------------------------------------------

quality_df_11C = pd.DataFrame(rows_11C)

assert len(quality_df_11C) == 2198
assert quality_df_11C["ecg_id"].nunique() == 2198
assert np.isfinite(
    quality_df_11C.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

quality_df_11C.to_csv(
    QUALITY_OUTPUT_PATH_11C,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("QUALITY SUMMARY")
print("-" * 70)

summary_cols_11C = [
    "mean_lead_rms_mV",
    "mean_lead_range_mV",
    "mean_baseline_wander_ratio",
    "mean_high_frequency_ratio",
    "zero_rms_leads"
]

print(
    quality_df_11C[summary_cols_11C]
    .describe()
    .T
    .to_string()
)

print()
print("ECGs containing at least one zero-RMS lead:",
      int((quality_df_11C["zero_rms_leads"] > 0).sum()))

print()
print("Saved:")
print(QUALITY_OUTPUT_PATH_11C)

print()
print("=" * 70)
print("STEP 11C STATUS: PASS")
print("=" * 70)

STEP 11C — NATURAL PTB-XL SIGNAL-QUALITY CHARACTERIZATION

Test ECGs: 2198
Leads per ECG: 12
Sampling frequency: 500 Hz

Processed  100/2198 ECGs
Processed  200/2198 ECGs
Processed  300/2198 ECGs
Processed  400/2198 ECGs
Processed  500/2198 ECGs
Processed  600/2198 ECGs
Processed  700/2198 ECGs
Processed  800/2198 ECGs
Processed  900/2198 ECGs
Processed 1000/2198 ECGs
Processed 1100/2198 ECGs
Processed 1200/2198 ECGs
Processed 1300/2198 ECGs
Processed 1400/2198 ECGs
Processed 1500/2198 ECGs
Processed 1600/2198 ECGs
Processed 1700/2198 ECGs
Processed 1800/2198 ECGs
Processed 1900/2198 ECGs
Processed 2000/2198 ECGs
Processed 2100/2198 ECGs
Processed 2198/2198 ECGs

QUALITY SUMMARY
----------------------------------------------------------------------
                             count      mean       std       min       25%       50%       75%       max
mean_lead_rms_mV            2198.0  0.191988  0.072958  0.073396  0.147262  0.174757  0.215218  0.787280
mean_lead_range_mV          219

In [7]:
import os
import json
import numpy as np

# ============================================================
# STEP 11D — CONTROLLED NOISE CONSTRUCTION VALIDATION
# ============================================================

NOISE_RESULTS_PATH_11D = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

PROTOCOL_PATH_11D = os.path.join(
    NOISE_RESULTS_PATH_11D,
    "step11B_noise_robustness_protocol.json"
)

with open(PROTOCOL_PATH_11D, "r", encoding="utf-8") as f:
    protocol_11D = json.load(f)

SEED_11D = protocol_11D["random_seed"]

SNR_LEVELS_11D = [
    float(x)
    for x in protocol_11D["noise_levels_db"]
]

EXPECTED_SHAPE_11D = (12, 5000)


def add_gaussian_noise_at_snr_11D(
    x,
    snr_db,
    rng
):
    """
    Add zero-mean Gaussian noise whose power is chosen
    to achieve the requested signal-to-noise ratio.

    SNR(dB) = 10 * log10(signal_power / noise_power)

    x shape:
        (12, 5000)
    """

    x = np.asarray(x, dtype=np.float32)

    if x.shape != EXPECTED_SHAPE_11D:
        raise ValueError(
            f"Expected {EXPECTED_SHAPE_11D}, got {x.shape}"
        )

    if not np.isfinite(x).all():
        raise ValueError(
            "Input contains non-finite values."
        )

    signal_power = float(
        np.mean(
            np.square(
                x.astype(np.float64)
            )
        )
    )

    if signal_power <= 0:
        raise ValueError(
            "Signal power must be positive."
        )

    noise_power = (
        signal_power /
        (10.0 ** (snr_db / 10.0))
    )

    noise_std = np.sqrt(noise_power)

    noise = rng.normal(
        loc=0.0,
        scale=noise_std,
        size=x.shape
    ).astype(np.float32)

    noisy = x + noise

    return noisy


def measured_snr_db_11D(
    clean,
    noisy
):
    """
    Measure the achieved SNR directly from the generated
    perturbation.
    """

    clean = np.asarray(
        clean,
        dtype=np.float64
    )

    noisy = np.asarray(
        noisy,
        dtype=np.float64
    )

    noise = noisy - clean

    signal_power = np.mean(
        np.square(clean)
    )

    noise_power = np.mean(
        np.square(noise)
    )

    return float(
        10.0 *
        np.log10(
            signal_power /
            noise_power
        )
    )


# ------------------------------------------------------------
# Reproducibility and SNR validation
# ------------------------------------------------------------

rng_11D = np.random.default_rng(SEED_11D)

# Synthetic but deterministic validation waveform.
# This validates the noise-generation mathematics without
# touching the locked test set.
t_11D = np.arange(
    EXPECTED_SHAPE_11D[1],
    dtype=np.float32
) / 500.0

test_signal_11D = np.zeros(
    EXPECTED_SHAPE_11D,
    dtype=np.float32
)

for lead_idx in range(12):
    test_signal_11D[lead_idx] = (
        0.8 * np.sin(
            2.0 * np.pi * 1.2 * t_11D
        )
        + 0.2 * np.sin(
            2.0 * np.pi * 8.0 * t_11D
        )
    )

validation_rows_11D = []

for snr_db in SNR_LEVELS_11D:

    noisy_1_11D = add_gaussian_noise_at_snr_11D(
        test_signal_11D,
        snr_db,
        rng_11D
    )

    measured_1_11D = measured_snr_db_11D(
        test_signal_11D,
        noisy_1_11D
    )

    # Reset RNG and regenerate to verify reproducibility.
    rng_repeat_11D = np.random.default_rng(SEED_11D)

    # Advance the RNG identically through previous conditions.
    for previous_snr_11D in SNR_LEVELS_11D[
        :SNR_LEVELS_11D.index(snr_db)
    ]:
        _ = add_gaussian_noise_at_snr_11D(
            test_signal_11D,
            previous_snr_11D,
            rng_repeat_11D
        )

    noisy_2_11D = add_gaussian_noise_at_snr_11D(
        test_signal_11D,
        snr_db,
        rng_repeat_11D
    )

    reproducible_11D = np.array_equal(
        noisy_1_11D,
        noisy_2_11D
    )

    measured_2_11D = measured_snr_db_11D(
        test_signal_11D,
        noisy_2_11D
    )

    snr_error_11D = abs(
        measured_1_11D - snr_db
    )

    validation_rows_11D.append({
        "snr_db_requested": snr_db,
        "snr_db_measured": measured_1_11D,
        "snr_absolute_error_db": snr_error_11D,
        "reproducible": reproducible_11D,
        "finite_output": bool(
            np.isfinite(noisy_1_11D).all()
        ),
        "shape_correct": (
            noisy_1_11D.shape ==
            EXPECTED_SHAPE_11D
        )
    })

validation_df_11D = (
    __import__("pandas")
    .DataFrame(validation_rows_11D)
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(validation_df_11D) == len(
    SNR_LEVELS_11D
)

assert validation_df_11D[
    "finite_output"
].all()

assert validation_df_11D[
    "shape_correct"
].all()

assert validation_df_11D[
    "reproducible"
].all()

assert (
    validation_df_11D[
        "snr_absolute_error_db"
    ].max()
    < 0.1
)

# ------------------------------------------------------------
# Save validation record
# ------------------------------------------------------------

OUTPUT_PATH_11D = os.path.join(
    NOISE_RESULTS_PATH_11D,
    "step11D_noise_generation_validation.csv"
)

validation_df_11D.to_csv(
    OUTPUT_PATH_11D,
    index=False
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("STEP 11D — CONTROLLED NOISE CONSTRUCTION VALIDATION")
print("=" * 70)
print()

print("Noise type: Additive zero-mean Gaussian")
print("Random seed:", SEED_11D)
print("Input shape:", EXPECTED_SHAPE_11D)
print()

print("SNR VALIDATION")
print("-" * 70)

print(
    validation_df_11D.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print(
    "Maximum absolute SNR error:",
    f'{validation_df_11D["snr_absolute_error_db"].max():.6f} dB'
)

print(
    "All outputs finite:",
    validation_df_11D["finite_output"].all()
)

print(
    "All outputs correct shape:",
    validation_df_11D["shape_correct"].all()
)

print(
    "Noise generation reproducible:",
    validation_df_11D["reproducible"].all()
)

print()
print("Saved:")
print(OUTPUT_PATH_11D)

print()
print("=" * 70)
print("STEP 11D STATUS: PASS")
print("=" * 70)

STEP 11D — CONTROLLED NOISE CONSTRUCTION VALIDATION

Noise type: Additive zero-mean Gaussian
Random seed: 42
Input shape: (12, 5000)

SNR VALIDATION
----------------------------------------------------------------------
 snr_db_requested  snr_db_measured  snr_absolute_error_db  reproducible  finite_output  shape_correct
        30.000000        29.981005               0.018995          True           True           True
        20.000000        19.953925               0.046075          True           True           True
        10.000000         9.989955               0.010045          True           True           True
         0.000000         0.029615               0.029615          True           True           True

Maximum absolute SNR error: 0.046075 dB
All outputs finite: True
All outputs correct shape: True
Noise generation reproducible: True

Saved:
/content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness/step11D_noise_generation_validation.csv

STEP 11D STATUS

In [8]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    confusion_matrix
)

# ============================================================
# STEP 11E — FROZEN RESNET NOISE-ROBUSTNESS EVALUATION
# ============================================================

PROJECT_PATH_11E = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

RESULTS_PATH_11E = os.path.join(
    PROJECT_PATH_11E,
    "results"
)

NOISE_RESULTS_PATH_11E = os.path.join(
    RESULTS_PATH_11E,
    "noise_robustness"
)

CHECKPOINT_PATH_11E = os.path.join(
    RESULTS_PATH_11E,
    "deep_learning",
    "ecg_resnet1d_best_validation.pt"
)

SPLIT_PATH_11E = os.path.join(
    RESULTS_PATH_11E,
    "ptbxl_patient_independent_split.csv"
)

TARGETS_PATH_11E = os.path.join(
    RESULTS_PATH_11E,
    "ptbxl_diagnostic_targets.csv"
)

CACHE_PATH_11E = "/content/ptbxl_raw_ecg_cache"

OUTPUT_PATH_11E = os.path.join(
    NOISE_RESULTS_PATH_11E,
    "step11E_noise_robustness_results.csv"
)

TARGET_COLUMNS_11E = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

NOISE_CONDITIONS_11E = [
    ("clean", None),
    ("snr_30db", 30.0),
    ("snr_20db", 20.0),
    ("snr_10db", 10.0),
    ("snr_0db", 0.0)
]

SEED_11E = 42

# ------------------------------------------------------------
# Frozen model architecture from Notebook 10
# ------------------------------------------------------------

class ResidualBlock1D_11E(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=7,
            stride=stride,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm1d(
            out_channels
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False
        )

        self.bn2 = nn.BatchNorm1d(
            out_channels
        )

        if (
            stride != 1
            or in_channels != out_channels
        ):
            self.downsample = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm1d(
                    out_channels
                )
            )
        else:
            self.downsample = nn.Identity()

    def forward(self, x):

        identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out


class ECGResNet1D_11E(nn.Module):

    def __init__(
        self,
        n_leads=12,
        n_classes=5
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                n_leads,
                64,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )

        self.layer1 = nn.Sequential(
            ResidualBlock1D_11E(64, 64),
            ResidualBlock1D_11E(64, 64)
        )

        self.layer2 = nn.Sequential(
            ResidualBlock1D_11E(
                64,
                128,
                stride=2
            ),
            ResidualBlock1D_11E(
                128,
                128
            )
        )

        self.layer3 = nn.Sequential(
            ResidualBlock1D_11E(
                128,
                256,
                stride=2
            ),
            ResidualBlock1D_11E(
                256,
                256
            )
        )

        self.layer4 = nn.Sequential(
            ResidualBlock1D_11E(
                256,
                512,
                stride=2
            ),
            ResidualBlock1D_11E(
                512,
                512
            )
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(
            512,
            n_classes
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.global_pool(x)

        x = x.squeeze(-1)

        return self.fc(x)


# ------------------------------------------------------------
# Verify cache
# ------------------------------------------------------------

if not os.path.isdir(CACHE_PATH_11E):

    raise FileNotFoundError(
        f"Local PTB-XL cache not found: {CACHE_PATH_11E}"
    )

cache_files_11E = [
    f for f in os.listdir(CACHE_PATH_11E)
    if f.endswith(".npy")
]

print("=" * 70)
print("STEP 11E — FROZEN RESNET NOISE-ROBUSTNESS EVALUATION")
print("=" * 70)
print()
print("Local cache:", CACHE_PATH_11E)
print("Cached ECG files:", len(cache_files_11E))

if len(cache_files_11E) < 21799:

    raise RuntimeError(
        "Local cache appears incomplete. "
        f"Expected at least 21,799 files, found {len(cache_files_11E)}."
    )

# ------------------------------------------------------------
# Load frozen split and targets
# ------------------------------------------------------------

split_df_11E = pd.read_csv(
    SPLIT_PATH_11E
)

targets_df_11E = pd.read_csv(
    TARGETS_PATH_11E
)

test_ids_11E = (
    split_df_11E.loc[
        split_df_11E["split"] == "test",
        "ecg_id"
    ]
    .astype(int)
    .tolist()
)

assert len(test_ids_11E) == 2198
assert len(set(test_ids_11E)) == 2198

targets_df_11E["ecg_id"] = (
    targets_df_11E["ecg_id"]
    .astype(int)
)

test_targets_11E = (
    targets_df_11E[
        targets_df_11E["ecg_id"].isin(
            test_ids_11E
        )
    ]
    .copy()
)

test_targets_11E = (
    test_targets_11E
    .set_index("ecg_id")
    .loc[test_ids_11E]
    .reset_index()
)

assert len(test_targets_11E) == 2198

Y_test_11E = (
    test_targets_11E[
        TARGET_COLUMNS_11E
    ]
    .to_numpy(dtype=np.float32)
)

assert Y_test_11E.shape == (
    2198,
    5
)

# ------------------------------------------------------------
# Load frozen model
# ------------------------------------------------------------

device_11E = torch.device("cpu")

checkpoint_11E = torch.load(
    CHECKPOINT_PATH_11E,
    map_location=device_11E,
    weights_only=False
)

model_11E = ECGResNet1D_11E(
    n_leads=12,
    n_classes=5
)

model_11E.load_state_dict(
    checkpoint_11E["model_state_dict"]
)

model_11E.to(device_11E)
model_11E.eval()

parameter_count_11E = sum(
    p.numel()
    for p in model_11E.parameters()
)

assert parameter_count_11E == 8_739_973

assert checkpoint_11E["epoch"] == 17

# ------------------------------------------------------------
# Noise helper
# ------------------------------------------------------------

def add_noise_11E(
    x,
    snr_db,
    rng
):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    if snr_db is None:
        return x.copy()

    signal_power = np.mean(
        np.square(
            x.astype(np.float64)
        )
    )

    if signal_power <= 0:
        raise ValueError(
            "Signal power must be positive."
        )

    noise_power = (
        signal_power /
        (10.0 ** (snr_db / 10.0))
    )

    noise_std = np.sqrt(
        noise_power
    )

    noise = rng.normal(
        0.0,
        noise_std,
        size=x.shape
    ).astype(np.float32)

    return x + noise


# ------------------------------------------------------------
# Metric helper
# ------------------------------------------------------------

def calculate_metrics_11E(
    y_true,
    probabilities
):

    rows = []

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    for class_idx, target in enumerate(
        TARGET_COLUMNS_11E
    ):

        y = y_true[:, class_idx]
        p = probabilities[:, class_idx]
        pred = predictions[:, class_idx]

        auroc = roc_auc_score(
            y,
            p
        )

        auprc = average_precision_score(
            y,
            p
        )

        f1 = f1_score(
            y,
            pred,
            zero_division=0
        )

        sensitivity = recall_score(
            y,
            pred,
            zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y,
            pred,
            labels=[0, 1]
        ).ravel()

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        )

        rows.append({
            "target": target,
            "AUROC": auroc,
            "AUPRC": auprc,
            "F1": f1,
            "sensitivity": sensitivity,
            "specificity": specificity
        })

    metrics_df = pd.DataFrame(rows)

    macro_row = {
        "target": "MACRO",
        "AUROC": metrics_df["AUROC"].mean(),
        "AUPRC": metrics_df["AUPRC"].mean(),
        "F1": metrics_df["F1"].mean(),
        "sensitivity": metrics_df["sensitivity"].mean(),
        "specificity": metrics_df["specificity"].mean()
    }

    metrics_df = pd.concat(
        [
            metrics_df,
            pd.DataFrame([macro_row])
        ],
        ignore_index=True
    )

    return metrics_df


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

all_results_11E = []

print()
print("MODEL")
print("-" * 70)
print("Architecture: ECGResNet1D")
print("Parameters:", f"{parameter_count_11E:,}")
print("Checkpoint epoch:", checkpoint_11E["epoch"])
print("Device:", device_11E)
print()

start_total_11E = time.time()

for condition_idx, (
    condition,
    snr_db
) in enumerate(
    NOISE_CONDITIONS_11E
):

    print("=" * 70)
    print(
        f"CONDITION {condition_idx + 1}/"
        f"{len(NOISE_CONDITIONS_11E)}: "
        f"{condition}"
    )
    print("=" * 70)

    rng_11E = np.random.default_rng(
        SEED_11E
    )

    probabilities_11E = []

    condition_start_11E = time.time()

    with torch.no_grad():

        for i, ecg_id in enumerate(
            test_ids_11E,
            start=1
        ):

            cache_path_11E = os.path.join(
                CACHE_PATH_11E,
                f"{ecg_id}.npy"
            )

            if not os.path.exists(
                cache_path_11E
            ):
                raise FileNotFoundError(
                    f"Missing cached ECG: {ecg_id}"
                )

            x = np.load(
                cache_path_11E
            )

            x = np.asarray(
                x,
                dtype=np.float32
            )

            # Cache is expected to contain
            # (12, 5000).
            if x.shape == (5000, 12):
                x = x.T

            if x.shape != (12, 5000):
                raise ValueError(
                    f"ECG {ecg_id} has shape "
                    f"{x.shape}, expected (12, 5000)."
                )

            if not np.isfinite(x).all():
                raise ValueError(
                    f"ECG {ecg_id} contains "
                    "non-finite values."
                )

            x_noisy = add_noise_11E(
                x,
                snr_db,
                rng_11E
            )

            tensor = torch.from_numpy(
                x_noisy
            ).unsqueeze(0)

            logits = model_11E(
                tensor
            )

            probabilities = torch.sigmoid(
                logits
            ).cpu().numpy()[0]

            probabilities_11E.append(
                probabilities
            )

            if (
                i % 250 == 0
                or i == len(test_ids_11E)
            ):

                elapsed = (
                    time.time()
                    - condition_start_11E
                )

                print(
                    f"Processed {i:4d}/"
                    f"{len(test_ids_11E)} "
                    f"| elapsed {elapsed:.1f}s"
                )

    probabilities_11E = np.asarray(
        probabilities_11E,
        dtype=np.float64
    )

    assert probabilities_11E.shape == (
        2198,
        5
    )

    assert np.isfinite(
        probabilities_11E
    ).all()

    metrics_11E = calculate_metrics_11E(
        Y_test_11E,
        probabilities_11E
    )

    condition_time_11E = (
        time.time()
        - condition_start_11E
    )

    print()
    print(
        metrics_11E.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    for _, metric_row in metrics_11E.iterrows():

        all_results_11E.append({
            "condition": condition,
            "snr_db": snr_db,
            "target": metric_row["target"],
            "AUROC": metric_row["AUROC"],
            "AUPRC": metric_row["AUPRC"],
            "F1": metric_row["F1"],
            "sensitivity": metric_row["sensitivity"],
            "specificity": metric_row["specificity"],
            "evaluation_time_seconds":
                condition_time_11E
        })

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results_df_11E = pd.DataFrame(
    all_results_11E
)

assert len(results_df_11E) == 30

results_df_11E.to_csv(
    OUTPUT_PATH_11E,
    index=False
)

total_time_11E = (
    time.time()
    - start_total_11E
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("STEP 11E — FINAL ROBUSTNESS SUMMARY")
print("=" * 70)

macro_results_11E = (
    results_df_11E[
        results_df_11E["target"] == "MACRO"
    ]
    .copy()
)

print()
print(
    macro_results_11E[
        [
            "condition",
            "snr_db",
            "AUROC",
            "AUPRC",
            "F1",
            "sensitivity",
            "specificity"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print(
    "Total evaluation time:",
    f"{total_time_11E / 60:.2f} minutes"
)

print()
print("Saved:")
print(OUTPUT_PATH_11E)

print()
print("=" * 70)
print("STEP 11E STATUS: PASS")
print("=" * 70)

FileNotFoundError: Local PTB-XL cache not found: /content/ptbxl_raw_ecg_cache